# 🎬 Arte Mediathek × Letterboxd
### Spielfilme in Originalversion mit deutschen Untertiteln — nach Letterboxd-Bewertung sortiert

**Ablauf:**
1. Arte-Filme von der MediathekViewWeb-API laden
2. Beste Version pro Film auswählen (`(Originalversion mit Untertitel)` bevorzugt, sonst deutsche Fassung)
3. Für jeden Film: TMDb-ID holen → Letterboxd-Seite über TMDb-ID aufrufen → Metadaten parsen
4. Alles zusammenführen und nach Letterboxd-Bewertung sortieren

**Voraussetzungen:**
```
pip install requests pandas beautifulsoup4 lxml
```
**Benötigte Accounts:**
- [themoviedb.org](https://www.themoviedb.org/settings/api) → kostenloser API-Key (v3)

In [33]:
# Einmalig ausführen falls Pakete noch nicht installiert sind
# %pip install requests pandas beautifulsoup4 lxml

In [34]:
import requests
import json
import re
import time
import urllib.parse
from datetime import datetime
from collections import defaultdict

import pandas as pd
from bs4 import BeautifulSoup
from IPython.display import display, HTML

from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Farben für die drei Bewertungsstufen ───────────────────────────────
FARBE_GUT       = "D6EFD8"   # Hellgrün  — 3.66 bis 4.0
FARBE_SEHR_GUT  = "A8D5A2"   # Mittelgrün — über 4.0
FARBE_OK        = "FFF3CD"   # Hellgelb  — 3.4 bis 3.65

# ── Konfiguration ──────────────────────────────────────────────────────────
# Maximale Anzahl Filme für die Letterboxd-Abfrage (für Tests klein halten)
MAX_FILMS = None

# Pause zwischen Letterboxd-Anfragen — bitte nicht unter 0.5 setzen
LETTERBOXD_DELAY = 0.8

# Mindestlaufzeit in Sekunden (3600 = 60 Minuten → schließt Kurzfilme aus)
MIN_DURATION_SEC = 3600

# TMDb API-Key (kostenlos von themoviedb.org/settings/api)
TMDB_KEY = "c0d452e595235cb23464d05a2283939e"

print("✓ Imports und Konfiguration geladen")

✓ Imports und Konfiguration geladen


---
## Schritt 1: Arte-Filmliste von MediathekViewWeb laden

**MediathekViewWeb** aggregiert alle öffentlich-rechtlichen Mediatheken und bietet eine Such-API.

Wir filtern nach:
- **Sender:** `ARTE.DE`
- **Thema:** enthält `"film"` → trifft `Kino - Filme`, `Fernsehfilm`, etc.
- **Mindestdauer:** 3600 Sekunden (60 Min.) → schließt Kurzfilme aus

> ⚠️ **Wichtig:** `Content-Type` muss `text/plain` sein, sonst antwortet die API mit 400.

In [ ]:
def lade_mediathek_filme(sender, size=500, duration_min=MIN_DURATION_SEC):
    """
    Lädt Filme von MediathekViewWeb.
    Gibt eine Liste von Einträgen zurück (jeder Eintrag = eine Versionsvariant eines Films).
    """
    query = {
        "queries": [
            {"fields": ["channel"], "query": sender},
            {"fields": ["topic"],   "query": "film"}
        ],
        "sortBy":       "timestamp",
        "sortOrder":    "desc",
        "future":       False,
        "offset":       0,
        "size":         size,
        "duration_min": duration_min
    }

    antwort = requests.post(
        "https://mediathekviewweb.de/api/query",
        data=json.dumps(query),
        headers={"Content-Type": "text/plain"},  # MUSS text/plain sein
        timeout=30
    )
    antwort.raise_for_status()

    data = antwort.json()
    if data.get("err"):
        raise ValueError(f"API-Fehler: {data['err']}")

    result_info = data.get("result", {})
    total  = result_info.get("queryInfo", {}).get("totalResults", "?")
    filme  = result_info.get("results", [])

    print(f"Treffer gesamt in API : {total}")
    print(f"Zurückgegeben         : {len(filme)}")
    return filme


rohdaten = lade_mediathek_filme('ARTE.DE')

### ✅ Sanity Check 1: Hat die API sinnvolle Daten geliefert?

In [36]:
# Prüfe ob Daten vorhanden und vollständig sind
assert len(rohdaten) > 0, "API hat 0 Ergebnisse zurückgegeben — Verbindung prüfen"

# Zeige alle vorhandenen Suffixe (wichtig für den nächsten Schritt)
alle_suffixe = set()
for film in rohdaten:
    m = re.search(r'\(([^)]+)\)\s*$', film['title'])
    alle_suffixe.add(m.group(1) if m else "[kein Suffix]")

print(f"✓ {len(rohdaten)} Einträge geladen")
print(f"\nVorhandene Titel-Suffixe: {sorted(alle_suffixe)}")
print(f"\nBeispiel-Einträge (erste 6 Titel):")
for f in rohdaten[:6]:
    print(f"  {f['title']}")

✓ 172 Einträge geladen

Vorhandene Titel-Suffixe: ['Audiodeskription', 'Originalversion', 'Originalversion mit Untertitel', '[kein Suffix]', 'mit Untertitel']

Beispiel-Einträge (erste 6 Titel):
  Parasite (Audiodeskription)
  Parasite
  Parasite (mit Untertitel)
  BlackBerry - Klick einer Generation (Audiodeskription)
  BlackBerry - Klick einer Generation (mit Untertitel)
  BlackBerry - Klick einer Generation


---
## Schritt 2: Beste Version pro Film auswählen

Arte stellt denselben Film in bis zu vier Varianten bereit — alle als separate Einträge in der API:

| Suffix | Bedeutung | Wollen wir? |
|---|---|---|
| `(Originalversion mit Untertitel)` | Originalsprache + deutsche UT | ✅ **1. Wahl** |
| *(kein Suffix)* | Deutsche Synchronfassung | ✅ Fallback |
| `(mit Untertitel)` | Deutsche Synchro + Untertitel für Gehörlose | ❌ |
| `(Audiodeskription)` | Deutsche Synchro + Audiobeschreibung | ❌ |
| `(Originalversion)` | Originalsprache **ohne** Untertitel | ❌ |

> **Strategie:** Für jeden Basis-Titel wählen wir genau eine Version aus — Priorität oben nach unten.

> **`⚠️ Geändert gegenüber ursprünglichem Notebook`**: `dedupliziere()` und `ist_omu()` werden durch
> `waehle_beste_version()` ersetzt, das beides in einem Schritt erledigt.

In [37]:
def extrahiere_basis(titel):
    """
    Entfernt den Versions-Suffix in Klammern am Titelende.
    'Minari (Originalversion mit Untertitel)' → 'Minari'
    """
    return re.sub(
        r'\s*\((Originalversion mit Untertitel|Originalversion|mit Untertitel|Audiodeskription)\)\s*$',
        '',
        titel,
        flags=re.IGNORECASE
    ).strip()


def versionstyp(titel):
    """
    Klassifiziert einen Titel nach Versionstyp.

    Rückgabewerte:
        'original'  → (Originalversion mit Untertitel)  ← bevorzugt
        'deutsch'   → kein Suffix                       ← Fallback
        'skip'      → alles andere                      ← wird ignoriert
    """
    if re.search(r'\(Originalversion mit Untertitel\)', titel, re.IGNORECASE):
        return 'original'
    elif re.search(r'\(mit Untertitel\)|\(Audiodeskription\)|\(Originalversion\)', titel, re.IGNORECASE):
        return 'skip'
    else:
        return 'deutsch'   # kein Suffix = deutsche Synchronfassung


def bereinige_titel(titel):
    """
    Entfernt Suffix für externe Suchanfragen (TMDb, Letterboxd).
    Identisch mit extrahiere_basis(), als eigene Funktion für Klarheit.
    """
    return extrahiere_basis(titel)


def waehle_beste_version(filme):
    """
    Ersetzt dedupliziere() + ist_omu() aus dem ursprünglichen Notebook.

    Für jeden Basis-Titel:
      1. (Originalversion mit Untertitel) wenn vorhanden
      2. sonst: kein Suffix (deutsche Fassung)
      3. alle anderen Varianten werden verworfen

    Bei mehreren Einträgen desselben Typs (Mehrfachausstrahlung)
    wird der neueste (höchster timestamp) behalten.
    """
    gruppen = defaultdict(dict)  # {basis_titel: {'original': film, 'deutsch': film}}

    for film in filme:
        basis = extrahiere_basis(film['title'])
        typ   = versionstyp(film['title'])

        if typ == 'skip':
            continue

        # Neuesten Eintrag bevorzugen
        vorhandener = gruppen[basis].get(typ)
        if vorhandener is None or film['timestamp'] > vorhandener['timestamp']:
            gruppen[basis][typ] = film

    ergebnis = []
    stats = {'original': 0, 'deutsch': 0}

    for basis, versionen in gruppen.items():
        if 'original' in versionen:
            ergebnis.append(versionen['original'])
            stats['original'] += 1
        elif 'deutsch' in versionen:
            ergebnis.append(versionen['deutsch'])
            stats['deutsch'] += 1

    print(f"Rohdaten gesamt              : {len(filme)}")
    print(f"Eindeutige Basis-Titel       : {len(gruppen)}")
    print(f"  davon Originalversion+UT   : {stats['original']}")
    print(f"  davon deutsche Fassung     : {stats['deutsch']}")
    print(f"  davon verworfen (skip)     : {len(filme) - sum(stats.values())}")
    return ergebnis


gefilterte_filme = waehle_beste_version(rohdaten)

Rohdaten gesamt              : 172
Eindeutige Basis-Titel       : 88
  davon Originalversion+UT   : 43
  davon deutsche Fassung     : 45
  davon verworfen (skip)     : 84


### ✅ Sanity Check 2: Wurde die richtige Version pro Film gewählt?

In [38]:
assert len(gefilterte_filme) > 0, "Keine Filme nach Versionsfilter — waehle_beste_version() prüfen"

# Überprüfe: Gibt es noch unerwünschte Versionen?
unerwuenscht = [f for f in gefilterte_filme
                if re.search(r'\(mit Untertitel\)|\(Audiodeskription\)|\(Originalversion\)(?! mit)',
                             f['title'], re.IGNORECASE)]
assert len(unerwuenscht) == 0, f"Unerwünschte Versionen noch vorhanden: {[f['title'] for f in unerwuenscht]}"

# Zeige Beispiele beider Typen
original_bsp = [f for f in gefilterte_filme if '(Originalversion mit Untertitel)' in f['title']][:4]
deutsch_bsp  = [f for f in gefilterte_filme if '(Originalversion mit Untertitel)' not in f['title']][:4]

print(f"✓ {len(gefilterte_filme)} Filme nach Versionsfilter — alle Versionen korrekt\n")
print("Beispiele — Originalversion mit Untertitel:")
for f in original_bsp:
    print(f"  ✓ {f['title']}")
print("\nBeispiele — Deutsche Fassung (kein OmU verfügbar):")
for f in deutsch_bsp:
    print(f"  · {f['title']}")

✓ 88 Filme nach Versionsfilter — alle Versionen korrekt

Beispiele — Originalversion mit Untertitel:
  ✓ Schock (Originalversion mit Untertitel)
  ✓ Santosh (Originalversion mit Untertitel)
  ✓ Benedetta (Originalversion mit Untertitel)
  ✓ Die Sirene (Originalversion mit Untertitel)

Beispiele — Deutsche Fassung (kein OmU verfügbar):
  · Parasite
  · BlackBerry - Klick einer Generation
  · Es war einmal in Amerika
  · Ich habe keine Angst


---
## Schritt 3: Letterboxd-Daten via TMDb holen

Letterboxd lädt seine Suchergebnisse per JavaScript — einfaches Scraping der Suchseite funktioniert nicht.

**Lösung in zwei Schritten:**

1. **TMDb API** → sucht nach deutschem Titel → liefert `tmdb_id` + Originaltitel + Beschreibung
2. **Letterboxd TMDb-Redirect** → `letterboxd.com/tmdb/{id}/` leitet direkt zur Filmseite weiter  
   → Filmseite enthält JSON-LD mit Bewertung, Regie, Jahr (server-seitig gerendert, kein JS)

> **`⚠️ Geändert gegenüber ursprünglichem Notebook`**: `suche_lb_slug()` und `hole_lb_filmseite()`  
> werden durch einen neuen dreistufigen Ablauf ersetzt. Letterboxd-Suche entfällt komplett.

In [39]:
# Browser-ähnliche Session — reduziert Bot-Erkennung bei Letterboxd
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "de-DE,de;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
})


def hole_tmdb(deutscher_titel):
    """
    Sucht einen Film auf TMDb anhand des deutschen Titels.

    Rückgabe: (tmdb_id, original_title, jahr, deutsche_beschreibung)
    oder      (None, None, None, None) wenn nicht gefunden.

    TMDb findet "Der Mann aus Marmor" und gibt uns tmdb_id=9452 zurück,
    mit der wir direkt auf letterboxd.com/tmdb/9452/ zugreifen können.
    """
    r = requests.get(
        "https://api.themoviedb.org/3/search/movie",
        params={
            "api_key":  TMDB_KEY,
            "query":    deutscher_titel,
            "language": "de-DE",   # Suche auf Deutsch → bessere Trefferquote
        },
        timeout=10
    )
    r.raise_for_status()
    ergebnisse = r.json().get("results", [])

    if not ergebnisse:
        return None, None, None, None

    treffer = ergebnisse[0]
    return (
        treffer["id"],
        treffer.get("original_title", deutscher_titel),
        str(treffer.get("release_date", ""))[:4],
        treffer.get("overview", ""),
    )


print("✓ Session und hole_tmdb() bereit")

✓ Session und hole_tmdb() bereit


In [40]:
# Diese drei Funktionen gehören alle in eine Zelle

def hole_lb_filmseite_von_html(html, url):
    soup = BeautifulSoup(html, "lxml")
    for script in soup.find_all("script", type="application/ld+json"):
        raw = script.string or ""
        raw = re.sub(r'/\*.*?\*/', '', raw, flags=re.DOTALL).strip()
        try:
            data = json.loads(raw)
        except (json.JSONDecodeError, TypeError):
            continue
        if data.get("@type") != "Movie":
            continue
        regie = ", ".join(
            d.get("name", "") for d in data.get("director", []) if d.get("name")
        )
        rat = data.get("aggregateRating", {})
        return {
            "lb_slug":     url.rstrip("/").split("/")[-1],
            "lb_title":    data.get("name", ""),
            "lb_year":     str(data.get("datePublished", ""))[:4],
            "lb_director": regie,
            "lb_desc":     data.get("description", ""),
            "lb_rating":   rat.get("ratingValue"),
            "lb_votes":    rat.get("ratingCount"),
            "lb_url":      url,
        }
    return None


def hole_lb_ueber_tmdb(tmdb_id):
    url = f"https://letterboxd.com/tmdb/{tmdb_id}/"
    try:
        r = session.get(url, timeout=15, allow_redirects=True)
        r.raise_for_status()
    except requests.RequestException:
        return None
    return hole_lb_filmseite_von_html(r.text, r.url)


def hole_lb_daten(film):
    titel = bereinige_titel(film.get("title", ""))
    time.sleep(0.3)
    tmdb_id, _, _, tmdb_desc = hole_tmdb(titel)
    if not tmdb_id:
        return {}
    time.sleep(LETTERBOXD_DELAY)
    lb = hole_lb_ueber_tmdb(tmdb_id)
    if lb and not lb.get("lb_desc"):
        lb["lb_desc"] = tmdb_desc
    return lb or {}


print("✓ Letterboxd-Funktionen bereit")

✓ Letterboxd-Funktionen bereit


### ✅ Sanity Check 3: Funktioniert die TMDb → Letterboxd Pipeline?

Teste mit einem einzigen bekannten Film bevor die vollständige Abfrage läuft.

In [41]:
# Test mit einem Film der definitiv auf Letterboxd ist
testfilm = {"title": "Minari (Originalversion mit Untertitel)", "duration": 6534}

titel = bereinige_titel(testfilm["title"])
print(f"Titel nach bereinige_titel(): '{titel}'")

tmdb_id, orig_titel, jahr, tmdb_desc = hole_tmdb(titel)
print(f"TMDb-ID     : {tmdb_id}")
print(f"Originaltitel: {orig_titel}")
print(f"Jahr        : {jahr}")

if tmdb_id:
    lb = hole_lb_ueber_tmdb(tmdb_id)
    if lb:
        print(f"\n✓ Letterboxd-Daten gefunden:")
        print(f"  Titel   : {lb['lb_title']}")
        print(f"  Regie   : {lb['lb_director']}")
        print(f"  Bewertung: ★ {lb['lb_rating']}")
        print(f"  URL     : {lb['lb_url']}")
    else:
        print("✗ Letterboxd: kein JSON-LD gefunden")
else:
    print("✗ TMDb: Film nicht gefunden — API-Key prüfen")

Titel nach bereinige_titel(): 'Minari'
TMDb-ID     : 615643
Originaltitel: Minari
Jahr        : 2021

✓ Letterboxd-Daten gefunden:
  Titel   : Minari
  Regie   : Lee Isaac Chung
  Bewertung: ★ 4.08
  URL     : https://letterboxd.com/film/minari/


---
## Schritt 4: Alle Filme abfragen

Jetzt wird für jeden Film in `gefilterte_filme` die komplette Pipeline durchlaufen.

> **Laufzeit:** ca. `MAX_FILMS × 1.1 Sekunden` (wegen Rate Limiting)  
> Bei `MAX_FILMS = 10` ca. 11 Sekunden.

In [42]:
# Sortiere nach Dauer (längste zuerst = wahrscheinlicher Spielfilm)
kandidaten = sorted(gefilterte_filme, key=lambda f: f.get("duration", 0), reverse=True)

if MAX_FILMS:
    kandidaten = kandidaten[:MAX_FILMS]

print(f"Starte Abfrage für {len(kandidaten)} Filme...")
print(f"Geschätzte Dauer: {len(kandidaten) * 1.1 / 60:.1f} Minuten\n")

ergebnisse = []

for i, film in enumerate(kandidaten, 1):
    titel = bereinige_titel(film.get("title", ""))
    dauer = f"{film.get('duration', 0) // 60} min"

    print(f"[{i:3d}/{len(kandidaten)}] {titel[:45]:45s} ({dauer})", end=" ")

    lb_daten = hole_lb_daten(film)

    if lb_daten:
        rating = lb_daten.get("lb_rating", "–")
        regie  = (lb_daten.get("lb_director") or "–")[:25]
        jahr   = lb_daten.get("lb_year", "–")
        print(f"→ ★ {rating}  {jahr}  {regie}")
    else:
        print("→ nicht gefunden")

    ergebnis = {
        "arte_titel":   film.get("title", ""),
        "arte_dauer":   film.get("duration", 0) // 60,
        "arte_datum":   datetime.fromtimestamp(film.get("timestamp", 0)).strftime("%d.%m.%Y"),
        "arte_thema":   film.get("topic", ""),
        "arte_url":     film.get("url_website", ""),
        **lb_daten,
    }
    ergebnisse.append(ergebnis)

gefunden = sum(1 for e in ergebnisse if e.get("lb_rating"))
print(f"\n✓ Fertig: {gefunden} von {len(ergebnisse)} Filmen auf Letterboxd gefunden")

Starte Abfrage für 88 Filme...
Geschätzte Dauer: 1.6 Minuten

[  1/88] Die Geheimnisse von Lissabon                  (249 min) → ★ 4.11    Raúl Ruiz
[  2/88] Es war einmal in Amerika                      (217 min) → ★ 4.22    Sergio Leone
[  3/88] Intime Geständnisse                           (193 min) → ★ 3.57    Liv Ullmann
[  4/88] Bis dann, mein Sohn                           (174 min) → ★ 4.16    Wang Xiaoshuai
[  5/88] Todesmelodie                                  (150 min) → ★ 3.96    Sergio Leone
[  6/88] Doch das Böse gibt es nicht                   (142 min) → ★ 3.8    Mohammad Rasoulof
[  7/88] Ein Gespräch mit... Walter Salles             (130 min) → nicht gefunden
[  8/88] Wie wilde Tiere                               (130 min) → ★ 4.08    Rodrigo Sorogoyen
[  9/88] Träum was Schönes                             (125 min) → ★ 3.36    Marco Bellocchio
[ 10/88] Memories of Murder                            (125 min) → ★ 4.42    Bong Joon Ho
[ 11/88] Captain Phillips          

## Lieblingsregisseure

In [43]:
# ── Lieblingsregisseure ────────────────────────────────────────────────
LIEBLINGSREGISSEURE = [
    "Almodóvar", "Almodovar",
    "Coen",
    "Lanthimos",
    "Paul Thomas Anderson",
    "Wim Wenders", "Wenders",
]

def ist_lieblingsregisseur(regie):
    if not regie:
        return False
    regie_lower = regie.lower()
    return any(name.lower() in regie_lower for name in LIEBLINGSREGISSEURE)


favoriten = [e for e in ergebnisse if ist_lieblingsregisseur(e.get("lb_director"))]

print(f"✓ {len(favoriten)} Filme von Lieblingsregisseuren gefunden:\n")
for f in sorted(favoriten, key=lambda e: e.get("lb_director", "")):
    print(f"  {f.get('lb_director'):25s}  ★ {f.get('lb_rating') or '–'}  {f.get('lb_title')}")

✓ 8 Filme von Lieblingsregisseuren gefunden:

  Joel Coen                  ★ 4.11  The Big Lebowski
  Paul Thomas Anderson       ★ 3.62  Hard Eight
  Pedro Almodóvar            ★ 3.69  High Heels
  Pedro Almodóvar            ★ 4.05  Pain and Glory
  Pedro Almodóvar            ★ 3.87  Talk to Her
  Pedro Almodóvar            ★ 3.96  Bad Education
  Pedro Almodóvar            ★ 3.7  Live Flesh
  Pedro Almodóvar            ★ 4.11  Women on the Verge of a Nervous Breakdown


---
## Schritt 5: Ergebnisse aufbereiten und anzeigen

In [44]:
# Nicht gefundene Filme entfernen (kein lb_rating = nicht auf Letterboxd)
ergebnisse = [e for e in ergebnisse if e.get("lb_rating")]

print(f"✓ {len(ergebnisse)} Filme mit Letterboxd-Bewertung behalten")

✓ 70 Filme mit Letterboxd-Bewertung behalten


In [49]:
df = pd.DataFrame(ergebnisse)

# Spalten-Mapping: interne Namen → Anzeigenamen
spalten_map = {
    "lb_title":    "Titel",
    "arte_titel":  "Arte-Titel",
    "lb_year":     "Jahr",
    "lb_director": "Regie",
    "lb_rating":   "LB ★",
    "lb_votes":    "Stimmen",
    "arte_dauer":  "Dauer (min)",
    "arte_datum":  "Arte-Datum",
    "lb_desc":     "Beschreibung",
    "lb_url":      "Letterboxd-URL",
    "arte_url":    "Arte-URL",
}

# Nur vorhandene Spalten umbenennen (verhindert KeyError wenn alle Filme leer)
vorhandene = {k: v for k, v in spalten_map.items() if k in df.columns}
df_anzeige = df[list(vorhandene.keys())].rename(columns=vorhandene)

# Sortieren nach LB-Bewertung — NaN ans Ende
if "LB ★" in df_anzeige.columns:
    df_anzeige = df_anzeige.sort_values("LB ★", ascending=False, na_position="last")

# Zahlenformatierung
if "LB ★" in df_anzeige.columns:
    df_anzeige["LB ★"] = df_anzeige["LB ★"].apply(
        lambda x: f"{x:.2f}" if pd.notna(x) else "–"
    )
if "Stimmen" in df_anzeige.columns:
    df_anzeige["Stimmen"] = df_anzeige["Stimmen"].apply(
        lambda x: f"{int(x):,}".replace(",", ".") if pd.notna(x) else "–"
    )
if "Beschreibung" in df_anzeige.columns:
    df_anzeige["Beschreibung"] = df_anzeige["Beschreibung"].apply(
        lambda x: (str(x)[:120] + "…") if pd.notna(x) and len(str(x)) > 120 else (x or "")
    )

# Kompakte Übersicht
anzeigespalten = [s for s in ["Titel", "Jahr", "Regie", "LB ★", "Stimmen", "Dauer (min)", "Arte-Datum"]
                  if s in df_anzeige.columns]

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.max_rows", None)
display(df_anzeige[anzeigespalten])

# Top 10 mit Beschreibung und klickbaren Links
hat_rating = df_anzeige["LB ★"] != "–" if "LB ★" in df_anzeige.columns else pd.Series([False]*len(df_anzeige))
top = df_anzeige[hat_rating].head(10)

if len(top) == 0:
    print("Noch keine Filme mit Letterboxd-Bewertung — MAX_FILMS erhöhen oder Abfrage wiederholen.")
else:
    zeilen = []
    for _, row in top.iterrows():
        titel    = row.get("Titel") or row.get("Arte-Titel", "?")
        arte_url = row.get("Arte-URL", "#")
        lb_url   = row.get("Letterboxd-URL", "#")
        zeilen.append(
            f"<tr>"
            f"<td><a href='{arte_url}' target='_blank'>{titel}</a></td>"
            f"<td>{row.get('Jahr','–')}</td>"
            f"<td>{row.get('Regie','–')}</td>"
            f"<td><a href='{lb_url}' target='_blank'>★ {row.get('LB ★','–')}</a></td>"
            f"<td>{row.get('Stimmen','–')}</td>"
            f"<td style='font-size:11px;color:#555'>{str(row.get('Beschreibung',''))[:100]}</td>"
            f"</tr>"
        )
    tabelle = (
        "<table border='1' cellpadding='6' style='border-collapse:collapse;font-size:13px'>"
        "<tr style='background:#222;color:#fff'>"
        "<th>Titel (→ Arte)</th><th>Jahr</th><th>Regie</th>"
        "<th>LB ★ (→ LB)</th><th>Stimmen</th><th>Beschreibung</th></tr>"
        + "".join(zeilen) + "</table>"
    )
    display(HTML(tabelle))

,Titel,Jahr,Regie,LB ★,Stimmen,Dauer (min),Arte-Datum
10,Parasite,,Bong Joon Ho,4.52,5.449.452,123,10.06.2026
8,Memories of Murder,,Bong Joon Ho,4.42,1.002.217,125,09.04.2018
1,Once Upon a Time in America,,Sergio Leone,4.22,310.457,217,08.06.2026
3,"So Long, My Son",,Wang Xiaoshuai,4.16,15.462,174,19.09.2022
23,The Big Lebowski,,Joel Coen,4.11,1.293.991,110,17.05.2026
61,Women on the Verge of a Nervous Breakdown,,Pedro Almodóvar,4.11,234.767,84,06.05.2019
0,Mysteries of Lisbon,,Raúl Ruiz,4.11,4.210,249,21.05.2026
58,Elevator to the Gallows,,Louis Malle,4.10,76.950,87,25.05.2026
31,Forever a Woman,,Kinuyo Tanaka,4.10,6.093,105,19.03.2026
6,The Beasts,,Rodrigo Sorogoyen,4.08,98.983,130,03.06.2026


Titel (→ Arte),Jahr,Regie,LB ★ (→ LB),Stimmen,Beschreibung
Parasite,,Bong Joon Ho,★ 4.52,5.449.452,"Familie Kim ist ganz unten angekommen: Vater, Mutter, Sohn und Tochter hausen in einem grünlich-schu"
Memories of Murder,,Bong Joon Ho,★ 4.42,1.002.217,In einem Dorf in Südkorea des Jahres 1986 treibt ein Frauenmörder sein Unwesen. Zwei Polizisten konn
Once Upon a Time in America,,Sergio Leone,★ 4.22,310.457,"New York zur Zeit der Prohibition. Max, Noodles und ihre Freunde verdienen sich bereits als Kinder m"
"So Long, My Son",,Wang Xiaoshuai,★ 4.16,15.462,"""Bis dann, mein Sohn"" ist meisterhaftes Kino, ein zutiefst berührendes Familienepos über Freundschaf"
The Big Lebowski,,Joel Coen,★ 4.11,1.293.991,"The Big Lebowski ist eigentlich ein Millonär aus LA, dessen Ehefrau Bunny hohe Geldschulden hat. Der"
Women on the Verge of a Nervous Breakdown,,Pedro Almodóvar,★ 4.11,234.767,Der Synchronsprecher Ivan hat sich von seiner langjährigen Freundin und Kollegin Pepa getrennt. Pepa
Mysteries of Lisbon,,Raúl Ruiz,★ 4.11,4.210,"Portugal, Anfang des 19. Jahrhunderts: Auf einem geistlichen Internat erfährt João, ein junger Waise"
Elevator to the Gallows,,Louis Malle,★ 4.10,76.950,Der ehemalige Fremdenlegionär Julien ermordet den Mann seiner Geliebten Florence in dessen Büro und
Forever a Woman,,Kinuyo Tanaka,★ 4.10,6.093,"Nachdem sich Fumiko von ihrem Ehemann hat scheiden lassen, wird ein Lesekreis ihr Refugium. Ermutigt"
The Beasts,,Rodrigo Sorogoyen,★ 4.08,98.983,Antoine (DENIS MÉNOCHET) und Olga (MARINA FOÏS) haben den Neuanfang gewagt. Das Ehepaar kehrte Frank


---
## Optional: Ergebnisse speichern

In [47]:
datei = f"arte_letterboxd_{datetime.now().strftime('%Y%m%d')}"

df_anzeige.to_csv(f"{datei}.csv", index=False, encoding="utf-8-sig")
print(f"✓ CSV gespeichert: {datei}.csv")

try:
    df_anzeige.to_excel(f"{datei}.xlsx", index=False)
    print(f"✓ Excel gespeichert: {datei}.xlsx")
except ImportError:
    print("ℹ  pip install openpyxl  für Excel-Export")

✓ CSV gespeichert: arte_letterboxd_20260612.csv
✓ Excel gespeichert: arte_letterboxd_20260612.xlsx


In [48]:
# Übersichtliche Excel-Tabelle

def bewertungsfarbe(rating):
    if rating is None:
        return None
    if rating > 4.0:
        return FARBE_SEHR_GUT
    elif rating >= 3.66:
        return FARBE_GUT
    elif rating >= 3.4:
        return FARBE_OK
    return None

# ── Spalten die gespeichert werden ─────────────────────────────────────
SPALTEN = [
    ("lb_title",    "Titel"),
    ("lb_year",     "Jahr"),
    ("lb_director", "Regie"),
    ("lb_rating",   "LB ★"),
    ("lb_votes",    "Stimmen"),
    ("arte_dauer",  "Dauer (min)"),
    ("arte_datum",  "Arte-Datum"),
    ("lb_desc",     "Beschreibung"),
    ("arte_url",    "Arte-Link"),
    ("lb_url",      "Letterboxd-Link"),
]

# ── Workbook aufbauen ──────────────────────────────────────────────────
wb = Workbook()
ws = wb.active
ws.title = "Arte × Letterboxd"

thin = Side(style="thin", color="CCCCCC")
rahmen = Border(left=thin, right=thin, top=thin, bottom=thin)

# Kopfzeile
header_fill = PatternFill("solid", fgColor="1C1C2E")
for col, (_, anzeigename) in enumerate(SPALTEN, 1):
    zelle = ws.cell(row=1, column=col, value=anzeigename)
    zelle.font      = Font(name="Arial", bold=True, color="F5C842", size=10)
    zelle.fill      = header_fill
    zelle.alignment = Alignment(horizontal="center", vertical="center")
    zelle.border    = rahmen

ws.row_dimensions[1].height = 22

# Datenzeilen — nach Bewertung sortiert
sortiert = sorted(ergebnisse, key=lambda e: e.get("lb_rating") or 0, reverse=True)

for zeile_nr, eintrag in enumerate(sortiert, 2):
    rating = eintrag.get("lb_rating")
    farbe  = bewertungsfarbe(rating)
    fill   = PatternFill("solid", fgColor=farbe) if farbe else None

    for col, (key, _) in enumerate(SPALTEN, 1):
        wert = eintrag.get(key, "")

        # Zahlenformatierung
        if key == "lb_rating" and wert:
            wert = round(float(wert), 2)
        elif key == "lb_votes" and wert:
            wert = int(wert)

        zelle = ws.cell(row=zeile_nr, column=col, value=wert)
        zelle.font      = Font(name="Arial", size=9)
        zelle.alignment = Alignment(vertical="center", wrap_text=(key == "lb_desc"))
        zelle.border    = rahmen
        if fill:
            zelle.fill = fill

        # Bewertungsspalte fett
        if key == "lb_rating":
            zelle.font = Font(name="Arial", size=9, bold=True)
            zelle.number_format = "0.00"

        # Stimmen mit Tausenderpunkt
        if key == "lb_votes":
            zelle.number_format = "#,##0"

    # Zeilenhöhe an Beschreibungslänge anpassen
# Excel-Zeilenhöhe in Punkten, ca. 13 Punkte pro Zeile bei Spaltenbreite 80
zeichen = len(str(eintrag.get("lb_desc") or ""))
zeilen  = max(1, -(-zeichen // 80))   # Aufrunden: wie viele Zeilen bei 80 Zeichen Breite
ws.row_dimensions[zeile_nr].height = zeilen * 13 + 4

# ── Spaltenbreiten ──────────────────────────────────────────────────────
breiten = {
    "lb_title":    28,
    "lb_year":      6,
    "lb_director": 22,
    "lb_rating":    8,
    "lb_votes":    12,
    "arte_dauer":   8,
    "arte_datum":  12,
    "lb_desc":     50,
    "arte_url":    14,
    "lb_url":      14,
}
for col, (key, _) in enumerate(SPALTEN, 1):
    ws.column_dimensions[get_column_letter(col)].width = breiten.get(key, 12)
    
# ── Autofilter + Fixierung ──────────────────────────────────────────────
ws.auto_filter.ref = f"A1:{get_column_letter(len(SPALTEN))}1"
ws.freeze_panes    = "A2"

# ── Legende ────────────────────────────────────────────────────────────
legende_zeile = len(sortiert) + 3
for farbe_hex, text in [
    (FARBE_SEHR_GUT, "★ > 4.0   — Sehr gut"),
    (FARBE_GUT,      "★ 3.66–4.0 — Gut"),
    (FARBE_OK,       "★ 3.4–3.65 — Okay"),
]:
    zelle = ws.cell(row=legende_zeile, column=1, value=text)
    zelle.fill = PatternFill("solid", fgColor=farbe_hex)
    zelle.font = Font(name="Arial", size=9)
    legende_zeile += 1

# ── Zweites Tabellenblatt: Lieblingsregisseure ─────────────────────────
ws2 = wb.create_sheet(title="Lieblingsregisseure")

# Kopfzeile (gleiche Spalten wie Haupttabelle)
for col, (_, anzeigename) in enumerate(SPALTEN, 1):
    zelle = ws2.cell(row=1, column=col, value=anzeigename)
    zelle.font      = Font(name="Arial", bold=True, color="F5C842", size=10)
    zelle.fill      = PatternFill("solid", fgColor="1C1C2E")
    zelle.alignment = Alignment(horizontal="center", vertical="center")
    zelle.border    = rahmen
ws2.row_dimensions[1].height = 22

# Daten — nach Regisseur dann Bewertung sortiert
favoriten_sortiert = sorted(
    favoriten,
    key=lambda e: (e.get("lb_director") or "", -(e.get("lb_rating") or 0))
)

for zeile_nr, eintrag in enumerate(favoriten_sortiert, 2):
    rating = eintrag.get("lb_rating")
    farbe  = bewertungsfarbe(rating)
    fill   = PatternFill("solid", fgColor=farbe) if farbe else None

    for col, (key, _) in enumerate(SPALTEN, 1):
        wert = eintrag.get(key, "")
        if key == "lb_rating" and wert:
            wert = round(float(wert), 2)
        elif key == "lb_votes" and wert:
            wert = int(wert)

        zelle = ws2.cell(row=zeile_nr, column=col, value=wert)
        zelle.font      = Font(name="Arial", size=9, bold=(key == "lb_rating"))
        zelle.alignment = Alignment(vertical="center", wrap_text=(key == "lb_desc"))
        zelle.border    = rahmen
        if fill:
            zelle.fill = fill
        if key == "lb_rating":
            zelle.number_format = "0.00"
        if key == "lb_votes":
            zelle.number_format = "#,##0"

    zeichen = len(str(eintrag.get("lb_desc") or ""))
    zeilen  = max(1, -(-zeichen // 45))
    ws2.row_dimensions[zeile_nr].height = zeilen * 15 + 6

# Spaltenbreiten
for col, (key, anzeigename) in enumerate(SPALTEN, 1):
    max_breite = len(anzeigename)
    for zeile_nr in range(2, len(favoriten_sortiert) + 2):
        wert = ws2.cell(row=zeile_nr, column=col).value
        if wert:
            max_breite = max(max_breite, min(len(str(wert)), 80))
    ws2.column_dimensions[get_column_letter(col)].width = max_breite + 2

ws2.auto_filter.ref = f"A1:{get_column_letter(len(SPALTEN))}1"
ws2.freeze_panes    = "A2"

print(f"✓ Tabellenblatt 'Lieblingsregisseure' mit {len(favoriten_sortiert)} Filmen erstellt")


# ── Speichern ───────────────────────────────────────────────────────────
dateiname = f"arte_letterboxd_{datetime.now().strftime('%Y%m%d')}.xlsx"
wb.save(dateiname)
print(f"✓ Gespeichert: {dateiname}")
print(f"  {len(sortiert)} Filme   |   "
      f"{sum(1 for e in sortiert if (e.get('lb_rating') or 0) > 4.0)} sehr gut   |   "
      f"{sum(1 for e in sortiert if 3.66 <= (e.get('lb_rating') or 0) <= 4.0)} gut   |   "
      f"{sum(1 for e in sortiert if 3.4 <= (e.get('lb_rating') or 0) < 3.66)} okay")

✓ Tabellenblatt 'Lieblingsregisseure' mit 8 Filmen erstellt
✓ Gespeichert: arte_letterboxd_20260612.xlsx
  70 Filme   |   11 sehr gut   |   20 gut   |   20 okay
